In [7]:
"""
================================================================================
 VERIFICACIÓN DE INTEGRIDAD – HASH DE STRAIN (OFICIAL vs LOCAL)
 ================================================================================
 Descarga el segmento GPS 1389379584 de GWOSC y calcula su hash MD5.
 Luego lee el archivo .hdf5 que tenías en /content/ (ajustar ruta si es necesario)
 y compara. Una diferencia confirma la corrupción de los archivos locales.
 ================================================================================
"""

import hashlib
import numpy as np

# 1. Descarga oficial
try:
    from gwpy.timeseries import TimeSeries
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gwpy"])
    from gwpy.timeseries import TimeSeries

print("Descargando segmento oficial GPS 1389379584 (32 s)...")
data_oficial = TimeSeries.fetch_open_data('H1', 1389379584, 1389379584+32, verbose=False)
hash_oficial = hashlib.md5(data_oficial.value.tobytes()).hexdigest()
print(f"Hash oficial: {hash_oficial}")

# 2. Lectura del archivo local (ajustá la ruta si lo tenés en otro lado)
ruta_local = "/content/H-H1_GWOSC_O4a_16KHZ_R1-1389379584-4096.hdf5"
try:
    import h5py
    with h5py.File(ruta_local, 'r') as f:
        strain_local = f['strain']['Strain'][:].astype(np.float64)
    # Tomamos solo los primeros 32 segundos para comparar la misma ventana
    fs_original = 16384  # frecuencia típica de los archivos de O4a
    muestras_32s = int(fs_original * 32)
    strain_local_32 = strain_local[:muestras_32s]
    hash_local = hashlib.md5(strain_local_32.tobytes()).hexdigest()
    print(f"Hash local  : {hash_local}")
    print("\n" + "="*50)
    if hash_oficial == hash_local:
        print("Los archivos coinciden. No hay evidencia de manipulación.")
    else:
        print("¡DISCREPANCIA DETECTADA! Los archivos locales están alterados.")
    print("="*50)
except FileNotFoundError:
    print(f"No se encontró el archivo local en {ruta_local}. Verificá la ruta.")

Descargando segmento oficial GPS 1389379584 (32 s)...
Hash oficial: 471a129d7a3c88c6ce28f1f05e82acd1
Hash local  : 023bf73611756d39e0dc1a0f174c82ba

¡DISCREPANCIA DETECTADA! Los archivos locales están alterados.
